# Test one fixed-lag motion feature

Question: does a one-second axis-vector change improve the retained full-HR ten-epoch recipe?
Reuse the completed seed-42 reference and train five new candidate folds locally.
Keep the folds, context, patching, width, dropout, loss, optimizer, budget, and checkpoint rule fixed.
Add one motion channel: seven raw channels become nine after the unchanged two-channel HR transform.
The patch input projection grows by 48 weights: 2,852 → 2,900 parameters.
This necessary input expansion makes this a feature-recipe comparison, not a capacity-matched mechanism test.
No class labels or prediction fields enter the added feature.
Keep all targets and the final test untouched. Read [the protocol](../docs/fixed-lag-motion-ablation.md).


## Locate the checkout and declare execution switches

**What:** Find the repository, configure logging, and declare optional setup, preparation, checks, and training.

**Why:** Existing local and cloned checkouts need the same paths, while model fitting requires explicit opt-in.


In [ ]:
import logging
import os
import shutil
import sys
from pathlib import Path

from IPython import get_ipython
from IPython.display import display

logging.basicConfig(level=logging.INFO, stream=sys.stdout, force=True)
logger = logging.getLogger("fixed_lag_motion_notebook")
ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents]
     if (path / "configs/fixed-lag-motion-grouped.yaml").exists()),
    Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", Path.cwd() / "sleep-classifier")),
).resolve()
ROOT = Path(os.environ.get("SLEEP_CLASSIFIER_ROOT", str(ROOT))).expanduser().resolve()

REPOSITORY_URL = "https://github.com/rbl-bidilatimotei/sleep-classifier.git"
BRANCH = "experiment/fixed-lag-motion"  # Select the PR branch before merge
RUN_SETUP = False
RUN_PREPARATION = False
RUN_CHECK = True
RUN_TRAINING = False

logger.info("Repository: %s", ROOT)
logger.info("Setup=%s preparation=%s preflight=%s training=%s",
            RUN_SETUP, RUN_PREPARATION, RUN_CHECK, RUN_TRAINING)


## Optional environment setup

**What:** Optionally clone this working branch and install Astral uv plus locked dependency groups.

**Why:** A fresh machine needs project code and the same dependencies. The existing local environment can skip this cell.
Fresh clones also require development features and the completed five-fold reference artifacts.


In [ ]:
if RUN_SETUP:
    if not ROOT.exists():
        !git clone --branch {BRANCH} {REPOSITORY_URL} "{ROOT}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Repository clone failed")
    if shutil.which("uv") is None:
        installer = ROOT / "artifacts/uv-install.sh"
        installer.parent.mkdir(parents=True, exist_ok=True)
        !curl -LsSf https://astral.sh/uv/install.sh -o "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv download failed")
        !sh "{installer}"
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("uv installation failed")
        os.environ["PATH"] = str(Path.home() / ".local/bin") + os.pathsep + os.environ["PATH"]
    %cd "{ROOT}"
    !uv sync --locked --group gru --group notebook --group dev
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Environment setup failed")
else:
    logger.info("Environment setup skipped")


## Verify the measured reference and controls

**What:** Validate the frozen participant folds, measured reference, and single added feature setting.

**Why:** A reused result is valid only when its measured recipe matches this comparison.
The audit is an immutable sensor source, not a source of labels or predictions for model inputs.


In [ ]:
import json

import pandas as pd

from sleep_classifier.experiment_evidence import EpochEvidenceReport
from sleep_classifier.grouped_validation import GroupedValidationExperiment
from sleep_classifier.neural_training import TrainingConfig
from sleep_classifier.paired_architecture_report import PairedArchitectureReport

experiment = GroupedValidationExperiment(ROOT / "configs/fixed-lag-motion-grouped.yaml")
reference = GroupedValidationExperiment(ROOT / "configs/transformer-window-local-hr-grouped.yaml")
plan = experiment.plan()
reference_summary = reference.output_dir / "summary.json"
reference_saved = json.loads(reference_summary.read_text())
reference_validation = PairedArchitectureReport(reference_summary, reference_summary).build()
reference_manifest = json.loads((reference.output_dir / "manifest.json").read_text())
resolved_device = str(TrainingConfig(experiment.model_settings).device())
if resolved_device != reference_manifest["device"]:
    raise ValueError("Candidate device differs from the measured reference")
if (reference_saved["plan"] != reference.plan()
        or plan["folds"] != reference_saved["plan"]["folds"]):
    raise ValueError("The participant-fold plan changed")
if len(plan["development"]) != 26 or len(plan["reserved_test"]) != 3:
    raise ValueError("The frozen cohort changed")

recipe = {k: v for k, v in experiment.model_settings.items() if k != "output_dir"}
original = {k: v for k, v in reference.model_settings.items() if k != "output_dir"}
if recipe.pop("motion_features") != {
    "name": "fixed_lag_motion", "audit_config": "motion-grid-audit.yaml",
} or recipe != original:
    raise ValueError("Only the fixed-lag motion input may change")
for result in reference_saved["results"]:
    path = Path(result["metrics_path"])
    path = path if path.is_absolute() else reference_summary.parent / path
    measured = json.loads(path.read_text())["config"]
    ignored = {"prepare_config", "output_dir", "device"}
    if ({k: v for k, v in measured.items() if k not in ignored}
            != {k: v for k, v in original.items() if k not in ignored}):
        raise ValueError("Reference YAML differs from its measured recipe")

display(pd.DataFrame(plan["folds"]).set_index("name"))
logger.info("Input change: 6 to 7 raw, 8 to 9 effective; parameters 2852 to 2900")
logger.info("Experiment: config=%s output=%s motion=%s masked_channels=%s",
            experiment.config_path, experiment.output_dir,
            experiment.model_settings["motion_features"],
            experiment.model_settings["window_features"].get("masked_channels", []))


## Inspect the sensor-only input contract

**What:** Load the verified cached audit and summarize motion support without raw sensor reads.

**Why:** Missing data must remain distinct from observed zero motion before training-only imputation.
Only the identity and motion value are joined into model rows. Labels and predictions are excluded.


In [ ]:
from sleep_classifier.causal_feature_audit import CausalFeatureAudit

audit_config = (
    experiment.model_path.parent / experiment.model_settings["motion_features"]["audit_config"]
)
audit = CausalFeatureAudit(audit_config)
audit_rows, audit_summary = audit.load_report()
display(pd.DataFrame(audit_summary["support_sensitivity"]))
logger.info("Verified motion source: %d targets; mean imputation uses training targets only",
            len(audit_rows))
logger.info("No support cutoff, missingness flag, target removal, or new class weighting is added")


## Prepare fresh candidate folds

**What:** Optionally copy the development features and bind the immutable motion snapshot to five new folds.

**Why:** New output paths protect prior runs. Preparation performs no optimization and opens no final-test observations.
Keep RUN_PREPARATION false after successful preparation; reruns validate existing fold inputs.


In [ ]:
if RUN_PREPARATION:
    grouped_config = experiment.config_path
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -m sleep_classifier.grouped_validation \
        --config "{grouped_config}" --prepare
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Preparation failed; inspect existing outputs before retrying")
else:
    logger.info("Motion-feature candidate preparation skipped")

## Check prepared folds without training

**What:** Run five no-gradient checks. Inspect saved history, normalization, input shape, output shape, and model size.

**Why:** Verify seven raw inputs, nine effective channels, and 2,900 candidate parameters without optimization.
The HR history remains ten epochs. The checkpoint must preserve both HR channels and the added motion definition.

In [ ]:
if RUN_CHECK and (experiment.output_dir / "manifest.json").exists():
    %cd "{ROOT}"
    for fold in plan["folds"]:
        fold_config = experiment.output_dir / fold["name"] / "gru.yaml"
        logger.info("Checking %s without optimization", fold["name"])
        !uv run --locked --group gru --group notebook python -m sleep_classifier.neural_training \
            --config "{fold_config}" --check
        if get_ipython().user_ns["_exit_code"] != 0:
            raise RuntimeError("Motion-feature candidate preflight failed")
elif RUN_CHECK:
    logger.info("Preflight pending: prepare the five folds first")
else:
    logger.info("Preflight skipped")

## Train only after explicit opt-in

**What:** Run five candidate fits using the validated config and shared readiness guard.

**Why:** Timo controls real-data optimization. Existing model results block accidental overwrites.
Keep RUN_TRAINING false for reports. Set it true only for the intended new local training run.


In [ ]:
if RUN_TRAINING:
    grouped_config = experiment.config_path
    logger.info("Training intent: config=%s output=%s history_epochs=%s masked_channels=%s",
                grouped_config, experiment.output_dir,
                experiment.model_settings["window_features"]["history_epochs"],
                experiment.model_settings.get("window_features", {}).get("masked_channels", []))
    experiment.validate_training_start()
    PairedArchitectureReport(reference_summary, reference_summary).build()
    if json.loads(reference_summary.read_text())["plan"] != reference.plan():
        raise ValueError("The reference participant-fold plan changed")
    %cd "{ROOT}"
    !uv run --locked --group gru --group notebook python -u -m sleep_classifier.grouped_validation \
        --config "{grouped_config}" --train
    if get_ipython().user_ns["_exit_code"] != 0:
        raise RuntimeError("Motion training failed; inspect retained evidence before retrying")
else:
    logger.info("Training skipped: enable RUN_TRAINING when ready")

## Inspect completion status and the measured reference

**What:** Show available reference/candidate fold status and the reference's pooled selected-checkpoint scores.

**Why:** A complete reference does not imply a complete candidate. Pending runs must not become fabricated performance scores.


In [ ]:
reporters = {"reference": EpochEvidenceReport(reference.output_dir),
             "candidate": EpochEvidenceReport(experiment.output_dir)}
evidence_by_role = {role: reporter.build() for role, reporter in reporters.items()}
status_rows = [{"recipe": role, "study_status": evidence["status"], "fold": fold["fold"],
                "fold_status": fold["status"], "recorded_epochs": len(fold["epochs"])}
               for role, evidence in evidence_by_role.items() for fold in evidence["folds"]]
if status_rows:
    display(pd.DataFrame(status_rows).set_index(["recipe", "fold"]))
else:
    logger.info("The candidate has not been prepared")
display(pd.DataFrame([{"recipe": "measured reference",
                       **reference_validation["reference"]}]).drop(
    columns=["confusion_matrix", "classes"],
))

## Inspect available learning curves

**What:** Plot saved training/evaluation losses, macro-F1, gradients, and learning rates.

**Why:** The derived channels may affect both stage decisions and probability fit. A lower NLL alone is not a useful classification gain.
Both recipes retain ordinary hard-label cross-entropy and the same checkpoint-selection rule.

In [ ]:
import matplotlib.pyplot as plt

for role, reporter in reporters.items():
    if not any(fold["epochs"] for fold in evidence_by_role[role]["folds"]):
        logger.info("%s has no saved per-epoch evidence; no curve is reconstructed", role)
        continue
    figure = reporter.plot_curves()
    figure.suptitle(f"{role}: completed-epoch evidence")
    figure.tight_layout(rect=(0, 0, 1, 0.97))
    display(figure)
    plt.close(figure)

## Compare selected checkpoints after all five fits finish

**What:** Verify matched targets, history definitions, normalization provenance, fixed controls, and selected scores.

**Why:** The explicit `motion_features` policy permits only the added motion input and its necessary projection weights.
Reference and candidate retain normalization origin, valid-pair count, and original heart scale. Derived mean and standard deviation may differ.
The primary outcome remains paired mean fold macro-F1. Validation also selects checkpoints, so this is development evidence.

In [ ]:
comparison = None
candidate_summary = experiment.output_dir / "summary.json"
if candidate_summary.exists() and json.loads(candidate_summary.read_text())["status"] == "complete":
    comparison = PairedArchitectureReport(
        reference_summary, candidate_summary, comparison="motion_features",
    ).build()
    display(pd.DataFrame(comparison["folds"]).set_index("fold"))
    score_rows = [{"recipe": role, "accuracy": comparison[role]["accuracy"],
                   "pooled_macro_f1": comparison[role]["macro_f1"]}
                  for role in ("reference", "candidate")]
    display(pd.DataFrame(score_rows).set_index("recipe"))
    class_rows = [{"recipe": role, "stage": stage, **scores}
                  for role in ("reference", "candidate")
                  for stage, scores in comparison[role]["classes"].items()]
    display(pd.DataFrame(class_rows).set_index(["recipe", "stage"]))
    destination = experiment.output_dir / "paired-comparison.json"
    destination.write_text(json.dumps(comparison, indent=2) + "\n")
    logger.info("Mean paired fold macro-F1 difference: %.6f",
                comparison["paired"]["mean_macro_f1_difference"])
else:
    logger.info("The motion-feature candidate is incomplete; no paired conclusion is available")

## Compare stage confusions

**What:** Plot reference and motion-candidate confusion matrices on the same held-out targets.

**Why:** Inspect REM, Wake, and Deep errors beside pooled scores. Rows are true stages; columns are predicted stages.

In [ ]:
if comparison is not None:
    import numpy as np
    from sklearn.metrics import ConfusionMatrixDisplay

    figure, axes = plt.subplots(1, 2, figsize=(11, 4))
    for axis, role in zip(axes, ("reference", "candidate"), strict=True):
        ConfusionMatrixDisplay(
            np.asarray(comparison[role]["confusion_matrix"]),
            display_labels=["Wake", "Light", "Deep", "REM"],
        ).plot(ax=axis, colorbar=False, values_format="d")
        axis.set_title(role)
    figure.tight_layout()
    display(figure)
    plt.close(figure)
else:
    logger.info("Paired confusion matrices are pending candidate completion")

## Inspect candidate selected-checkpoint probability loss

**What:** Read selected-epoch validation NLL and evaluation-mode training NLL, then calculate pooled validation NLL.

**Why:** Negative log-likelihood measures probability assigned to the true stage; lower is better.
A confident wrong prediction receives a large penalty. NLL complements hard-label F1 and does not select our checkpoints.
Weight each fold by target count. Keep partial evidence separate from a complete pooled score.
Both displayed NLL measurements use unsmoothed hard targets.

In [ ]:
candidate_rows = []
selected_loss_sum = 0.0
selected_target_count = 0
for fold in evidence_by_role["candidate"]["folds"]:
    selected = next((epoch for epoch in fold["epochs"]
                     if epoch["epoch"] == fold["best_epoch"]), None)
    if selected is None:
        continue
    count = sum(stage["count"] for stage in selected["validation_classes"].values())
    selected_target_count += count
    selected_loss_sum += selected["validation_nll"] * count
    candidate_rows.append({"fold": fold["fold"], "selected_epoch": fold["best_epoch"],
                           "selected_validation_nll": selected["validation_nll"],
                           "selected_training_evaluation_nll": selected["training_evaluation_nll"],
                           "scored_targets": count})
if candidate_rows:
    display(pd.DataFrame(candidate_rows).set_index("fold"))
    if len(candidate_rows) == 5:
        logger.info("Candidate pooled selected-checkpoint NLL: %.6f",
                    selected_loss_sum / selected_target_count)
    else:
        logger.info("Partial fold NLL only; no complete candidate aggregate yet")
else:
    logger.info("Selected-checkpoint NLL is pending candidate completion")

## Compare startup and transition slices

**What:** Compare paired scores by available raw history and true-stage transitions after training completes.

**Why:** Startup targets remain eligible despite unavailable derived history. Inspect supports and REM/Deep trade-offs beside macro-F1.
These post-hoc labels define evaluation slices only. They never become model inputs.
Use history bins 1–4, 5–29, and full context. The first bin has unavailable target-level history with minimum history four.
Both recipes use the same thirty-epoch context and ten-epoch HR reference. Motion is computed within each completed epoch.
Retain every eligible target. Do not average slice macro-F1 scores to reconstruct pooled macro-F1.

In [ ]:
from sleep_classifier.temporal_slice_report import TemporalSliceReport

if comparison is not None:
    temporal_report = TemporalSliceReport.from_grouped(
        reference_summary, candidate_summary, short_history_epochs=4,
        comparison="motion_features",
    ).build()
    slice_rows = []
    slice_class_rows = []
    for row in temporal_report["slices"]:
        for role in ("reference", "candidate"):
            scores = row[role]
            if scores is None:
                continue
            slice_rows.append({"slice": row["name"], "recipe": role,
                               "targets": row["targets"], "accuracy": scores["accuracy"],
                               "macro_f1": scores["macro_f1"]})
            slice_class_rows.extend(
                {"slice": row["name"], "recipe": role, "stage": stage, **values}
                for stage, values in scores["classes"].items()
            )
    display(pd.DataFrame(slice_rows).set_index(["slice", "recipe"]))
    display(pd.DataFrame(slice_class_rows).set_index(["slice", "recipe", "stage"]))
    (experiment.output_dir / "temporal-slices.json").write_text(
        json.dumps(temporal_report, indent=2) + "\n",
    )
else:
    logger.info("Temporal slices require five complete paired candidate fits")

## Inspect retained diagnostics

**What:** Summarize saved candidate epoch evidence with the reusable report command.

**Why:** Missing results must appear as pending, rather than as fabricated curves or scores.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python -m sleep_classifier.experiment_evidence \
    --output-dir "{experiment.output_dir}" --report
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Motion-feature evidence inspection failed")

## Export the executed HTML report

**What:** Execute report and preflight cells through a fresh kernel, then save local HTML.

**Why:** The source notebook remains output-free, while a reproducible executed report preserves inspection results.
Automated export skips setup, preparation, training, and this recursive export cell regardless of execution switches.


In [ ]:
%cd "{ROOT}"
!uv run --locked --group gru --group notebook python scripts/execute_notebook.py \
    --notebook notebooks/36_fixed_lag_motion.ipynb
if get_ipython().user_ns["_exit_code"] != 0:
    raise RuntimeError("Motion-feature notebook export failed")